# Time Series Regression with Signature Methods

This notebook demonstrates how to use the Nadaraya-Watson signature regressor for time series regression tasks and compares it with other baseline methods.

## Overview
- Load time series datasets from the Aeon library
- Augment time series with time dimension
- Apply signature-based regression
- Compare with DTW and signature-based baselines
- Evaluate model performance using MSE and R² metrics

## 1. Import Required Libraries

In [1]:
import sys
import os
# Add parent directory to path so we can import from nwsig package
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import matplotlib.pyplot as plt
import numpy as np
from aeon.datasets import load_regression
from aeon.regression.distance_based import KNeighborsTimeSeriesRegressor
from aeon.transformations.collection.signature_based import SignatureTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.utils import shuffle
from sklearn.preprocessing import MinMaxScaler

# Import from nwsig package
from nwsig import NWSignatureRegressor, augment_with_time

import warnings
warnings.filterwarnings('ignore')


import signature as efm_sig
from signature.path_signature import path_to_fm_signature
import jax.numpy as jnp

## 2. Helper Functions

## 3. ARCH/GARCH

Let's start with a single dataset to understand the workflow.

In [2]:
import numpy as np
from arch import arch_model
from sklearn.model_selection import train_test_split

np.random.seed(0)
ret = np.random.randn(1000)

am = arch_model(ret, vol="Garch", p=1, q=1)
res = am.fit(disp="off")

n_sims = 100
n_obs = 100
simulations = np.zeros((n_sims, n_obs))

for i in range(n_sims):
    sim = am.simulate(res.params, nobs=n_obs)
    simulations[i, :] = sim['data']

X = simulations[:,:-1]
y = simulations[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X Training size: {X_train.shape}, X Test size: {X_test.shape}")
print(f"y_train size: {y_train.shape}, y_test size: {y_test.shape}")

X Training size: (80, 99), X Test size: (20, 99)
y_train size: (80,), y_test size: (20,)


In [3]:

n_train, length = X_train.shape
n_test, _ = X_test.shape

time_vector = np.linspace(0, 1, length).reshape(1, length, 1)
time_channel_train = np.tile(time_vector, (n_train, 1, 1))
time_channel_test = np.tile(time_vector, (n_test, 1, 1))

X_train_augmented = np.concatenate([time_channel_train, X_train.reshape(n_train, length, 1)], axis=2)
X_test_augmented = np.concatenate([time_channel_test, X_test.reshape(n_test, length, 1)], axis=2)

print(f"Augmented training data shape: {X_train_augmented.shape}")
print(f"Augmented test data shape: {X_test_augmented.shape}")

Augmented training data shape: (80, 99, 2)
Augmented test data shape: (20, 99, 2)


## 4. Train NW Signature Regressor

We use cross-validation to automatically select the best bandwidth parameter.

In [4]:
# Extract initial conditions
#X0_train = X_train_augmented[:, 0, :]
#X0_test = X_test_augmented[:, 0, :]

X0_train = np.zeros_like(X_train_augmented[:, 0, :])
X0_test = np.zeros_like(X_test_augmented[:, 0, :])

# Initialize regressor with cross-validation
nw_regressor = NWSignatureRegressor(
    signature_level=3,
    input_type='paths',
    cv_folds=3,
    bandwidth_grid=np.linspace(0.001, 10, 50)
)

# Fit the model
print("Training NW Signature regressor...")
nw_regressor.fit(X_train_augmented, X0_train, y_train)
print(f"Best bandwidth: {nw_regressor.best_bandwidth_:.4f}")

Training NW Signature regressor...
Fitting NW estimator with input_type='paths'
Input shape: (80, 99, 2), target shape: (80,)
Target statistics: mean=-0.128861, std=1.073838, range=[-2.417432, 2.346772]
Computing signatures from paths...
Computing normal signature for path 1/80...
Computing normal signature for path 2/80...
Computing normal signature for path 3/80...
Computing normal signature for path 4/80...
Computing normal signature for path 5/80...
Computing normal signature for path 6/80...
Computing normal signature for path 7/80...
Computing normal signature for path 8/80...
Computing normal signature for path 9/80...
Computing normal signature for path 10/80...
Computing normal signature for path 11/80...
Computing normal signature for path 12/80...
Computing normal signature for path 13/80...
Computing normal signature for path 14/80...
Computing normal signature for path 15/80...
Computing normal signature for path 16/80...
Computing normal signature for path 17/80...
Comput

In [5]:
# Make predictions
y_pred_nw = nw_regressor.predict(X_test_augmented, X0_test)

# Evaluate performance
mse_nw = mean_squared_error(y_test, y_pred_nw)
rmse_nw = np.sqrt(mse_nw)
mae_nw = mean_absolute_error(y_test, y_pred_nw)
r2_nw = r2_score(y_test, y_pred_nw)

print(f"\nNW Signature Test Performance:")
print(f"  MSE:  {mse_nw:.4f}")
print(f"  RMSE: {rmse_nw:.4f}")
print(f"  MAE:  {mae_nw:.4f}")
print(f"  R²:   {r2_nw:.4f}")

Computing normal signature for path 1/20...
Computing normal signature for path 2/20...
Computing normal signature for path 3/20...
Computing normal signature for path 4/20...
Computing normal signature for path 5/20...
Computing normal signature for path 6/20...
Computing normal signature for path 7/20...
Computing normal signature for path 8/20...
Computing normal signature for path 9/20...
Computing normal signature for path 10/20...
Computing normal signature for path 11/20...
Computing normal signature for path 12/20...
Computing normal signature for path 13/20...
Computing normal signature for path 14/20...
Computing normal signature for path 15/20...
Computing normal signature for path 16/20...
Computing normal signature for path 17/20...
Computing normal signature for path 18/20...
Computing normal signature for path 19/20...
Computing normal signature for path 20/20...

NW Signature Test Performance:
  MSE:  0.7946
  RMSE: 0.8914
  MAE:  0.7030
  R²:   0.0236


## 4.bis Train NW EFM Signature Regressor

In [6]:
print([i for i in range(X_train.shape[1])])

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98]


In [7]:
# Extract initial conditions
#X0_train = X_train_augmented[:, 0, :]
#X0_test = X_test_augmented[:, 0, :]

X0_train = np.zeros_like(X_train_augmented[:, 0, :])
X0_test = np.zeros_like(X_test_augmented[:, 0, :])

# Initialize regressor with cross-validation
nw_regressor = NWSignatureRegressor(
    signature_level=3,
    input_type='paths',
    cv_folds=3,
    efm_signature=True,
    bandwidth_grid=np.linspace(0.001, 10, 50),
    lam = jnp.array([i for i in range(X_train.shape[1])])
)

# Fit the model
print("Training NW EFM Signature regressor...")
nw_regressor.fit(X_train_augmented, X0_train, y_train)
print(f"Best bandwidth: {nw_regressor.best_bandwidth_:.4f}")

Training NW EFM Signature regressor...
Fitting NW estimator with input_type='paths'
Input shape: (80, 99, 2), target shape: (80,)
Target statistics: mean=-0.128861, std=1.073838, range=[-2.417432, 2.346772]
Computing signatures from paths...
Computing EFM signature for path 1/80...
Computing EFM signature for path 2/80...
Computing EFM signature for path 3/80...
Computing EFM signature for path 4/80...
Computing EFM signature for path 5/80...
Computing EFM signature for path 6/80...
Computing EFM signature for path 7/80...
Computing EFM signature for path 8/80...
Computing EFM signature for path 9/80...
Computing EFM signature for path 10/80...
Computing EFM signature for path 11/80...
Computing EFM signature for path 12/80...
Computing EFM signature for path 13/80...
Computing EFM signature for path 14/80...
Computing EFM signature for path 15/80...
Computing EFM signature for path 16/80...
Computing EFM signature for path 17/80...
Computing EFM signature for path 18/80...
Computing E

In [8]:
# Make predictions
y_pred_nw_efm = nw_regressor.predict(X_test_augmented, X0_test)

# Evaluate performance
mse_nw_efm = mean_squared_error(y_test, y_pred_nw)
rmse_nw_efm = np.sqrt(mse_nw)
mae_nw_efm = mean_absolute_error(y_test, y_pred_nw)
r2_nw_efm = r2_score(y_test, y_pred_nw)

print(f"\nNW Signature Test Performance:")
print(f"  MSE:  {mse_nw:.4f}")
print(f"  RMSE: {rmse_nw:.4f}")
print(f"  MAE:  {mae_nw:.4f}")
print(f"  R²:   {r2_nw:.4f}")

Computing EFM signature for path 1/20...
Computing EFM signature for path 2/20...
Computing EFM signature for path 3/20...
Computing EFM signature for path 4/20...
Computing EFM signature for path 5/20...
Computing EFM signature for path 6/20...
Computing EFM signature for path 7/20...
Computing EFM signature for path 8/20...
Computing EFM signature for path 9/20...
Computing EFM signature for path 10/20...
Computing EFM signature for path 11/20...
Computing EFM signature for path 12/20...
Computing EFM signature for path 13/20...
Computing EFM signature for path 14/20...
Computing EFM signature for path 15/20...
Computing EFM signature for path 16/20...
Computing EFM signature for path 17/20...
Computing EFM signature for path 18/20...
Computing EFM signature for path 19/20...
Computing EFM signature for path 20/20...

NW Signature Test Performance:
  MSE:  0.7946
  RMSE: 0.8914
  MAE:  0.7030
  R²:   0.0236


### 5.1 DTW with Dynamic Time Warping (1-NN DTWD)

In [9]:
# DTW-based regressor
print("Training 1-NN DTW regressor...")
dtw_regressor = KNeighborsTimeSeriesRegressor(n_neighbors=1, distance="dtw")
dtw_regressor.fit(X_train, y_train)

# Predict
y_pred_dtw = dtw_regressor.predict(X_test)
mse_dtw = mean_squared_error(y_test, y_pred_dtw)
rmse_dtw = np.sqrt(mse_dtw)
r2_dtw = r2_score(y_test, y_pred_dtw)

print(f"DTW Test Performance:")
print(f"  MSE:  {mse_dtw:.4f}")
print(f"  RMSE: {rmse_dtw:.4f}")
print(f"  R²:   {r2_dtw:.4f}")

Training 1-NN DTW regressor...
DTW Test Performance:
  MSE:  3.2290
  RMSE: 1.7969
  R²:   -2.9679


### 5.2 DTWA (DTW with Averaging/Weighting)

In [10]:
# DTWA-based regressor (weighted DTW)
print("Training 1-NN DTWA regressor...")
dtwa_regressor = KNeighborsTimeSeriesRegressor(n_neighbors=1, distance="wdtw")
dtwa_regressor.fit(X_train, y_train)

# Predict
y_pred_dtwa = dtwa_regressor.predict(X_test)
mse_dtwa = mean_squared_error(y_test, y_pred_dtwa)
rmse_dtwa = np.sqrt(mse_dtwa)
r2_dtwa = r2_score(y_test, y_pred_dtwa)

print(f"DTWA Test Performance:")
print(f"  MSE:  {mse_dtwa:.4f}")
print(f"  RMSE: {rmse_dtwa:.4f}")
print(f"  R²:   {r2_dtwa:.4f}")

Training 1-NN DTWA regressor...
DTWA Test Performance:
  MSE:  3.2838
  RMSE: 1.8121
  R²:   -3.0353


### 5.3 Signature + Random Forest (SigP)

In [11]:
# Signature-based features with Random Forest
import iisignature
print("Training Signature + Random Forest regressor...")



def compute_signature(X, level=3):
    n_samples, length = X.shape
    signatures = []
    for i in range(n_samples):
        signatures.append(iisignature.sig(X[i,:].reshape(-1,1), level))
    return np.array(signatures)

# Transform data
print(f"Raw data shape: {X_train.shape}")
X_train_sig = compute_signature(X_train)
X_test_sig = compute_signature(X_test)
print(f"Signature shape: {X_train_sig.shape}")

# Train Random Forest
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)
rf_regressor.fit(X_train_sig, y_train)

# Predict
y_pred_sigp = rf_regressor.predict(X_test_sig)
mse_sigp = mean_squared_error(y_test, y_pred_sigp)
rmse_sigp = np.sqrt(mse_sigp)
r2_sigp = r2_score(y_test, y_pred_sigp)

print(f"Signature+RF Test Performance:")
print(f"  MSE:  {mse_sigp:.4f}")
print(f"  RMSE: {rmse_sigp:.4f}")
print(f"  R²:   {r2_sigp:.4f}")

Training Signature + Random Forest regressor...
Raw data shape: (80, 99)
Signature shape: (80, 3)
Signature+RF Test Performance:
  MSE:  1.2764
  RMSE: 1.1298
  R²:   -0.5684


### 5.4 Summary of Single Dataset Comparison

In [12]:
# Comparison summary
print("\n" + "="*80)
print(f"COMPARISON ON GARCH(1,1) SIMULATED DATA")
print("="*80)
print(f"{'Method':<30} {'MSE':<15} {'RMSE':<15} {'R²':<10}")
print("-"*80)
print(f"{'NW Signature':<30} {mse_nw:<15.4f} {rmse_nw:<15.4f} {r2_nw:<10.4f}")
print(f"{'NW Signature (EFM)':<30} {mse_nw:<15.4f} {rmse_nw:<15.4f} {r2_nw:<10.4f}")
print(f"{'1-NN DTW':<30} {mse_dtw:<15.4f} {rmse_dtw:<15.4f} {r2_dtw:<10.4f}")
print(f"{'1-NN DTWA':<30} {mse_dtwa:<15.4f} {rmse_dtwa:<15.4f} {r2_dtwa:<10.4f}")
print(f"{'Signature + Random Forest':<30} {mse_sigp:<15.4f} {rmse_sigp:<15.4f} {r2_sigp:<10.4f}")


COMPARISON ON GARCH(1,1) SIMULATED DATA
Method                         MSE             RMSE            R²        
--------------------------------------------------------------------------------
NW Signature                   0.7946          0.8914          0.0236    
NW Signature (EFM)             0.7946          0.8914          0.0236    
1-NN DTW                       3.2290          1.7969          -2.9679   
1-NN DTWA                      3.2838          1.8121          -3.0353   
Signature + Random Forest      1.2764          1.1298          -0.5684   


## 6. Batch Evaluation on Multiple Datasets

Evaluate all methods on several benchmark datasets.

In [13]:
# Select regression datasets for demonstration
datasets = [
    "Covid3Month",
    "AppliancesEnergy",
    #"BeijingPM10Quality",
    #"BeijingPM25Quality",
    #"BIDMC32HR"
]

In [14]:
# Run all methods on multiple datasets
results = {}

for dataset_name in datasets:
    print(f"\n{'='*60}")
    print(f"Processing: {dataset_name}")
    print('='*60)
    
    try:
        # Load data
        X_train, y_train = load_regression(dataset_name, split="train")
        X_test, y_test = load_regression(dataset_name, split="test")
        
        # Shuffle
        X_train, y_train = shuffle(X_train, y_train, random_state=42)
        X_test, y_test = shuffle(X_test, y_test, random_state=42)
        
        print(f"Data shape: {X_train.shape}")
        print(f"Target range: [{y_train.min():.2f}, {y_train.max():.2f}]")
        
        results[dataset_name] = {}
        
        # 1. NW Signature Regressor
        try:
            X_train_augmented = augment_with_time(X_train)
            X_test_augmented = augment_with_time(X_test)

            # Scale Train data
            X_train_augmented = scaler.fit_transform(
                X_train_augmented.reshape(-1, X_train_augmented.shape[-1])
            ).reshape(X_train_augmented.shape)

            # Scale Test data
            X_test_augmented = scaler.fit_transform(
                X_test_augmented.reshape(-1, X_test_augmented.shape[-1])
            ).reshape(X_test_augmented.shape)

            # Extract initial conditions
            X0_train = X_train_augmented[:, 0, :]
            X0_test = X_test_augmented[:, 0, :]
            
            nw_reg = NWSignatureRegressor(
                signature_level=3,
                input_type='paths',
                cv_folds=3,
                bandwidth_grid=np.linspace(0.01, 2.0, 30)
            )
            nw_reg.fit(X_train_augmented, X0_train, y_train)
            y_pred = nw_reg.predict(X_test_augmented,X0_test)
            
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[dataset_name]['NW Signature'] = {'mse': mse, 'rmse': rmse, 'r2': r2}
            print(f"  NW Signature - MSE: {mse:.4f}, R²: {r2:.4f}")
        except Exception as e:
            print(f"  NW Signature failed: {str(e)}")
            results[dataset_name]['NW Signature'] = None
        
        # 2. DTW
        try:
            dtw_reg = KNeighborsTimeSeriesRegressor(n_neighbors=1, distance="dtw")
            dtw_reg.fit(X_train, y_train)
            y_pred = dtw_reg.predict(X_test)
            
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[dataset_name]['DTW'] = {'mse': mse, 'rmse': rmse, 'r2': r2}
            print(f"  DTW - MSE: {mse:.4f}, R²: {r2:.4f}")
        except Exception as e:
            print(f"  DTW failed: {str(e)}")
            results[dataset_name]['DTW'] = None
        
        # 3. DTWA
        try:
            dtwa_reg = KNeighborsTimeSeriesRegressor(n_neighbors=1, distance="wdtw")
            dtwa_reg.fit(X_train, y_train)
            y_pred = dtwa_reg.predict(X_test)
            
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[dataset_name]['DTWA'] = {'mse': mse, 'rmse': rmse, 'r2': r2}
            print(f"  DTWA - MSE: {mse:.4f}, R²: {r2:.4f}")
        except Exception as e:
            print(f"  DTWA failed: {str(e)}")
            results[dataset_name]['DTWA'] = None
        
        # 4. Signature + Random Forest
        try:
            sig_transform = SignatureTransformer(
                augmentation_list=("addtime",),
                window_name="global",
                sig_tfm="signature",
                depth=3,
            )
            X_train_sig = sig_transform.fit_transform(X_train)
            X_test_sig = sig_transform.transform(X_test)
            
            rf_reg = RandomForestRegressor(n_estimators=100, random_state=42)
            rf_reg.fit(X_train_sig, y_train)
            y_pred = rf_reg.predict(X_test_sig)
            
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)
            
            results[dataset_name]['Sig+RF'] = {'mse': mse, 'rmse': rmse, 'r2': r2}
            print(f"  Sig+RF - MSE: {mse:.4f}, R²: {r2:.4f}")
        except Exception as e:
            print(f"  Sig+RF failed: {str(e)}")
            results[dataset_name]['Sig+RF'] = None
        
    except Exception as e:
        print(f"Error loading {dataset_name}: {str(e)}")
        results[dataset_name] = {'NW Signature': None, 'DTW': None, 'DTWA': None, 'Sig+RF': None}


Processing: Covid3Month
Data shape: (140, 1, 84)
Target range: [0.00, 0.18]
  NW Signature failed: name 'scaler' is not defined
  DTW - MSE: 0.0031, R²: -0.5576
  DTWA - MSE: 0.0025, R²: -0.2488
  Sig+RF failed: SignatureTransformer requires package 'esig' to be present in the python environment, but 'esig' was not found. 'esig' is a soft dependency and not included in the base aeon installation. Please run: `pip install esig` to install the esig package. To install all soft dependencies, run: `pip install aeon[all_extras]`

Processing: AppliancesEnergy
Data shape: (95, 24, 144)
Target range: [5.34, 26.28]
  NW Signature failed: name 'scaler' is not defined
  DTW - MSE: 27.3183, R²: -1.3107


KeyboardInterrupt: 